# Across-night movement and HR response — one subject

All saved crops represent sleep period time (SPT), including wake within that period. No further sleep/wake selection is applied. The SPT denominator is the span of saved acceleration timestamps plus one sample period; exact diary boundaries are not saved in these exports. Acquisition gaps remain in SPT, are reported as reduced coverage, and are never bridged by filtering or HR windows.

Movement AUC is treated primarily as a **continuous variable on a log10 scale**. AUC tertiles are retained only for descriptive visualization/backward comparison; their common 1/3 and 2/3 cutoffs are fitted across all candidate bursts and therefore can change when nights are added. Movement summaries include every detected burst; HR-response analyses include only eligible isolated events.

HR follows the existing notebook: **FIT-recorded HR**, baseline −20 to −6 s, full epoch −20 to +49 s, peak increase = max(0, maximum post-onset percentage change), latency to positive peak in +1 to +49 s. No positive peak means undefined latency. HR timing is approximate. Recovery is deliberately omitted.

Nights receive equal weight in across-night metric means and HR curves. Shading is between-night ±1 SD, describing variability in this subject, not population uncertainty. Correlations are descriptive; bursts cluster within nights and chronological nights may also be dependent. No independent-event p-values are reported.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

MODULE_DIR = next(candidate for base in [Path.cwd(), *Path.cwd().parents]
                  for candidate in [base, base / "offline_processing"]
                  if (candidate / "across_nights_v2.py").exists())
sys.path.insert(0, str(MODULE_DIR))
from across_nights_v2 import analyze_subject

# Override with OVERNIGHT_PROCESSED_ROOT or edit this cell for another subject.
PROCESSED_ROOT = Path(os.environ.get("OVERNIGHT_PROCESSED_ROOT", str(Path.home() / "Documents/garmin/sleep/garmin_processed")))
NAIVE_TIMEZONE = "Europe/Rome"  # Saved exports contain naive local wall time.
OUTPUT_DIR = MODULE_DIR / "outputs" / "across_nights"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
COLORS = {"Low": "#167D8D", "Medium": "#A6761D", "High": "#B52E55"}

In [ ]:
result = analyze_subject(PROCESSED_ROOT, naive_timezone=NAIVE_TIMEZONE)
nights = result["nights"]
nightly = result["nightly"]
print(f"{len(nights)} nights; pooled AUC cutoffs: {result['cutoffs']} g·s")
display(nights)
display(result["events"].loc[lambda x: ~x.included].groupby(["night", "exclusion_reason"]).size().rename("excluded"))

## Nightly metrics and trends

Dates identify night folders by recording start. Points/lines connect nightly means in chronological order; the dashed line is the equal-night mean. Movement burden is the sum of detected burst durations divided by total SPT, including the gaps within merged bursts as defined by the existing detector. Missing group means stay missing, not zero. Counts, coverage, and HR eligibility should be checked before interpreting trends.

In [ ]:
metrics = {"p2p_mean_g": "Mean burst p2p (g)", "auc_mean_g_s": "Mean burst AUC (g·s)",
           "duration_mean_s": "Mean burst duration (s)", "burst_spt_pct": "Burst duration / SPT (%)",
           "hr_peak_mean_pct": "Mean HR peak increase (%)", "hr_latency_mean_s": "Mean peak latency (s)"}
# summary = nightly.groupby("group", sort=False)[list(metrics)].agg(["mean", "std", "count"])
# display(summary)
# display(nightly)
# fig, axes = plt.subplots(3, 2, figsize=(14, 12), constrained_layout=True)
# for ax, (metric, label) in zip(axes.flat, metrics.items()):
#     for group, color in {"All": "#333333", **COLORS}.items():
#         data = nightly.loc[nightly.group == group].set_index("night").reindex(nights.index)
#         dates = pd.to_datetime(data.index, format="%Y-%m-%d-%H-%M-%S")
#         ax.plot(dates, data[metric], 'o-', color=color, label=group, alpha=.85)
#     overall = nightly.loc[nightly.group == "All", metric].mean()
#     ax.axhline(overall, color="black", linestyle="--", alpha=.4)
#     ax.set(ylabel=label)
#     ax.tick_params(axis="x", rotation=45)
# axes.flat[0].legend(ncol=4)
# plt.show()

nightly_all = nightly.loc[nightly["group"] == "All"]

fig, axes = plt.subplots(3, 2, figsize=(14, 12), constrained_layout=True)

for ax, (metric, label) in zip(axes.flat, metrics.items()):
    data = (
        nightly_all
        .set_index("night")
        .reindex(nights.index)
    )

    dates = pd.to_datetime(data.index, format="%Y-%m-%d-%H-%M-%S")

    ax.plot(
        dates,
        data[metric],
        "o-",
        color="#333333",
        label="All",
        alpha=0.85
    )

    overall = nightly_all[metric].mean()
    ax.axhline(overall, color="black", linestyle="--", alpha=0.4)

    ax.set_ylabel(label)
    ax.tick_params(axis="x", rotation=45)

axes.flat[0].legend()
plt.show()

## 1. Continuous movement intensity -> HR response

The primary movement predictor is now **continuous AUC**, shown on a logarithmic axis rather than treated as three biological classes. Tertiles remain useful for the familiar response-curve plots below, but the dose-response analysis does not depend on them.

The black line is a descriptive pooled fit of HR peak increase against `log10(AUC)`. The large points are quantile-bin means +/- SEM to make the shape of the relationship visible without imposing tertile boundaries. Events are clustered within nights, so this plot is descriptive rather than an independent-event inferential model.

In [ ]:
events_cont = result["events"].reset_index()
eligible_cont = (
    events_cont.loc[events_cont["included"] & (events_cont["AUC"] > 0)]
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["AUC", "hr_peak_increase_pct"])
)
bins = result["continuous_auc_bins"]
global_reactivity = result["global_reactivity"]

fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
ax.scatter(
    eligible_cont["AUC"], eligible_cont["hr_peak_increase_pct"],
    s=14, alpha=0.13, color="#555555", label="Eligible movement events"
)

ax.errorbar(
    bins["auc_median_g_s"], bins["hr_mean_pct"], yerr=bins["hr_sem_pct"],
    fmt="o-", linewidth=2, capsize=3, color="black", label="Quantile-bin mean +/- SEM"
)

if len(eligible_cont):
    x_grid = np.geomspace(eligible_cont["AUC"].min(), eligible_cont["AUC"].max(), 250)
    y_grid = (
        global_reactivity["hr_at_reference_auc_pct"]
        + global_reactivity["reactivity_slope_pct_per_decade"]
          * (np.log10(x_grid) - global_reactivity["reference_log10_auc"])
    )
    ax.plot(x_grid, y_grid, linewidth=2.4, label="Pooled log-AUC fit")

# Old tertile cutoffs shown only as a visual reference.
for cutoff in result["cutoffs"]:
    ax.axvline(cutoff, color="gray", linestyle="--", alpha=0.35)

ax.set_xscale("log")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set(
    xlabel="Movement AUC (g*s; logarithmic scale)",
    ylabel="Post-onset HR peak increase (%)",
    title="Continuous movement intensity and cardiovascular response",
)
ax.legend()
plt.show()

display(global_reactivity.to_frame("value"))

## 2. Nightly movement-adjusted cardiac reactivity

For each night, eligible events are fitted as:

`HR peak increase (%) = intercept + slope * log10(AUC / reference AUC)`

where the **reference AUC is the subject-wide median eligible movement AUC**. This gives two complementary nightly parameters:

- **Reactivity slope**: HR percentage-point change associated with a 10-fold increase in movement AUC.
- **HR response at reference movement**: expected HR peak response for the same standardized movement intensity every night.

The latter is especially useful for distinguishing a night with stronger cardiovascular responses from a night that simply contained larger movements. Error bars are classical within-night OLS SEs and should be interpreted descriptively because events within a night are not independent physiological replicates.

In [ ]:
reactivity = result["nightly_reactivity"].reindex(nights.index)
dates = pd.to_datetime(reactivity.index, format="%Y-%m-%d-%H-%M-%S")
ref_auc = result["reference_auc_g_s"]

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True, constrained_layout=True)

axes[0].errorbar(
    dates,
    reactivity["reactivity_slope_pct_per_decade"],
    yerr=reactivity["reactivity_slope_se"],
    fmt="o-", capsize=3, color="#333333"
)
axes[0].axhline(reactivity["reactivity_slope_pct_per_decade"].mean(),
                color="black", linestyle="--", alpha=0.35)
axes[0].set_ylabel("Reactivity slope\n(%-points per 10x AUC)")
axes[0].set_title("Nightly cardiac reactivity to movement")

axes[1].errorbar(
    dates,
    reactivity["hr_at_reference_auc_pct"],
    yerr=reactivity["hr_at_reference_auc_se"],
    fmt="o-", capsize=3, color="#333333"
)
axes[1].axhline(reactivity["hr_at_reference_auc_pct"].mean(),
                color="black", linestyle="--", alpha=0.35)
axes[1].set_ylabel(f"Expected HR peak (%)\nat AUC={ref_auc:.3g} g*s")
axes[1].tick_params(axis="x", rotation=45)

plt.show()
display(reactivity)

## 3. Separate motor burden from cardiac reactivity

The same night can now be described on two different axes:

**Motor burden** uses *all detected bursts* and asks how much movement occurred (events/hour, total AUC/hour, movement duration/SPT).

**Cardiac reactivity** uses only HR-eligible isolated events and asks how the cardiovascular system responded *after accounting for movement intensity*.

This separation is important: a high unadjusted mean HR response can arise because movements were larger, because cardiovascular reactivity was greater, or both.

In [ ]:
phenotype = result["nightly_phenotype"].reindex(nights.index)
dates = pd.to_datetime(phenotype.index, format="%Y-%m-%d-%H-%M-%S")

fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)

# Motor-history descriptors: all detected bursts.
axes[0, 0].plot(dates, phenotype["bursts_per_hour"], "o-", color="#333333")
axes[0, 0].set_ylabel("Bursts / hour")
axes[0, 0].set_title("Motor burden: event frequency")

axes[0, 1].plot(dates, phenotype["auc_per_hour_g_s"], "o-", color="#333333")
axes[0, 1].set_ylabel("Total movement AUC / hour (g*s/h)")
axes[0, 1].set_title("Motor burden: integrated movement")

# Cardiac-history descriptors: adjusted for event movement intensity.
axes[1, 0].errorbar(
    dates, phenotype["hr_at_reference_auc_pct"],
    yerr=phenotype["hr_at_reference_auc_se"], fmt="o-", capsize=3, color="#333333"
)
axes[1, 0].set_ylabel("HR peak at reference movement (%)")
axes[1, 0].set_title("Standardized cardiovascular response")

axes[1, 1].errorbar(
    dates, phenotype["reactivity_slope_pct_per_decade"],
    yerr=phenotype["reactivity_slope_se"], fmt="o-", capsize=3, color="#333333"
)
axes[1, 1].set_ylabel("Reactivity slope (%-points / 10x AUC)")
axes[1, 1].set_title("Movement-response gain")

for ax in axes.flat:
    ax.tick_params(axis="x", rotation=45)

plt.show()

# Optional diagnostic: does standardized cardiac response simply track motor burden?
fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
ax.scatter(phenotype["auc_per_hour_g_s"], phenotype["hr_at_reference_auc_pct"], s=45)
for night, row in phenotype.iterrows():
    if np.isfinite(row.get("auc_per_hour_g_s", np.nan)) and np.isfinite(row.get("hr_at_reference_auc_pct", np.nan)):
        ax.annotate(pd.to_datetime(night, format="%Y-%m-%d-%H-%M-%S").strftime("%m-%d"),
                    (row["auc_per_hour_g_s"], row["hr_at_reference_auc_pct"]),
                    xytext=(4, 3), textcoords="offset points", fontsize=8, alpha=0.7)
ax.set(
    xlabel="Motor burden: total movement AUC / hour (g*s/h)",
    ylabel="Cardiac response at reference movement (%)",
    title="Motor burden and movement-adjusted cardiac response",
)
plt.show()

display(phenotype[[
    "burst_count_all", "bursts_per_hour", "movement_burden_spt_pct", "auc_per_hour_g_s",
    "hr_eligible_n", "mean_hr_peak_pct_unadjusted", "hr_at_reference_auc_pct",
    "reactivity_slope_pct_per_decade", "reactivity_r2"
]])

## Full HR curves by common AUC tertile

Thin traces show each night's mean response. The thick curve averages those nightly means, giving each available night equal weight. Nights without eligible bursts in a tertile do not contribute. The table reports contributing nights and bursts.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True, constrained_layout=True)
for ax, (group, color) in zip(axes, COLORS.items()):
    local = result["night_curves"].query("group == @group")
    for night, curve in local.groupby("night"):
        ax.plot(curve.relative_s, curve.mean_pct, color=color, alpha=.16, linewidth=.8)
    grand = result["grand_curves"].query("group == @group")
    ax.plot(grand.relative_s, grand['mean'], color=color, linewidth=2.5)
    ax.fill_between(grand.relative_s, grand['mean'] - grand['std'], grand['mean'] + grand['std'], color=color, alpha=.18)
    ax.axvline(0, color="gray", linestyle="--")
    ax.axhline(0, color="gray", linewidth=.7)
    ax.axvspan(-20, -5, color="gray", alpha=.1)
    ax.set(title=f"{group} AUC — {local.night.nunique()} nights", xlabel="Seconds from burst onset")
axes[0].set_ylabel("HR change from baseline (%)")
plt.show()
display(nightly.pivot(index="night", columns="group", values="hr_n"))

## Movement–HR correlations: bursts and nightly averages

Both predictors use the same **AUC-defined** groups. Each burst plot pools eligible events (colour identifies night); each nightly point averages the movement metric and HR peak over the same eligible events within that size group. These answer different questions: event-level association versus covariation of nightly means. Pearson and Spearman coefficients are shown; fewer than three pairs or constant inputs yield missing coefficients. Pooled associations can reflect differences between nights.

In [ ]:
display(result["correlations"])
for level, table in [("Burst", result["burst_pairs"]), ("Night", result["night_pairs"])]:
    fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
    for row, group in enumerate(["Small", "Medium + large"]):
        subset = table.loc[table.size_group == group]
        for col, (metric, label) in enumerate([("AUC", "Movement AUC (g·s)"), ("peak-to-peak", "Movement p2p (g)")]):
            ax = axes[row, col]
            for i, night in enumerate(nights.index):
                points = subset.loc[subset.night == night]
                ax.scatter(points[metric], points.hr_peak_increase_pct, color=plt.cm.tab20(i % 20), alpha=.7, s=24)
            stats = result["correlations"].query("level == @level and group == @group and metric == @metric").iloc[0]
            ax.set(title=f"{level}: {group} | n={stats['n']} | r={stats.pearson_r:.2f}, ρ={stats.spearman_rho:.2f}",
                   xlabel=label, ylabel="HR peak increase (%)")
    plt.show()


## Local exports

Exports and executed notebook outputs contain personal data and remain in the Git-ignored outputs directory. Source notebooks are published with empty outputs. Re-running recomputes all nights and common thresholds.

In [ ]:
import json
for name in [
    "nights", "events", "nightly", "night_curves", "grand_curves", "night_pairs",
    "correlations", "continuous_auc_bins", "nightly_reactivity",
    "nightly_motor_burden", "nightly_phenotype"
]:
    result[name].to_csv(OUTPUT_DIR / f"{name}.csv", index=name in ["nights", "events"])

result["global_reactivity"].to_csv(OUTPUT_DIR / "global_reactivity.csv", header=True)

(OUTPUT_DIR / "settings.json").write_text(json.dumps({
    "nights": list(nights.index),
    "common_auc_cutoffs_g_s": list(result["cutoffs"]),
    "auc_primary_analysis": "continuous log10(AUC); tertiles descriptive only",
    "reference_auc_g_s": result["reference_auc_g_s"],
    "reactivity_slope_unit": "HR percentage-points per 10-fold AUC increase",
    "reactivity_intercept": "expected HR peak increase at subject-wide median eligible AUC",
    "naive_timezone": NAIVE_TIMEZONE, "sampling_rate_hz": 100, "threshold_g": .040,
    "max_accel_gap_s": .25, "merge_gap_s": 5, "hr_source": "FIT-recorded HR",
    "hr_limits_bpm": [30, 200], "max_hr_gap_s": 3, "isolation_s": 30,
    "exclude_late_overlap": True, "spt": "saved acceleration span plus sample period",
    "night_weighting": "equal", "recovery": False,
}, indent=2), encoding="utf-8")
print(f"Saved local tables to {OUTPUT_DIR}")